In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\pd413\data science\New project\data\india_housing_prices.csv")

df.head()

,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


In [3]:
print("shape:", df.shape)
df.info()

shape: (250000, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  object 
 2   City                            250000 non-null  object 
 3   Locality                        250000 non-null  object 
 4   Property_Type                   250000 non-null  object 
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  object 
 11  Floor_No                        250000 non-null  int64  
 

In [4]:
df.describe()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000
mean,125000.500000,2.999396,2749.813216,254.586854,0.130597,2006.520012,14.966800,15.503004,18.479988,5.499860,5.498016
std,72168.927986,1.415521,1300.606954,141.349921,0.130747,9.808575,8.948047,8.671618,9.808575,2.878639,2.871860
min,1.000000,1.000000,500.000000,10.000000,0.000000,1990.000000,0.000000,1.000000,2.000000,1.000000,1.000000
25%,62500.750000,2.000000,1623.000000,132.550000,0.050000,1998.000000,7.000000,8.000000,10.000000,3.000000,3.000000
50%,125000.500000,3.000000,2747.000000,253.870000,0.090000,2007.000000,15.000000,15.000000,18.000000,5.000000,5.000000
75%,187500.250000,4.000000,3874.000000,376.880000,0.160000,2015.000000,23.000000,23.000000,27.000000,8.000000,8.000000
max,250000.000000,5.000000,5000.000000,500.000000,0.990000,2023.000000,30.000000,30.000000,35.000000,10.000000,10.000000


In [5]:
df.isnull().sum()

ID                                0
State                             0
City                              0
Locality                          0
Property_Type                     0
BHK                               0
Size_in_SqFt                      0
Price_in_Lakhs                    0
Price_per_SqFt                    0
Year_Built                        0
Furnished_Status                  0
Floor_No                          0
Total_Floors                      0
Age_of_Property                   0
Nearby_Schools                    0
Nearby_Hospitals                  0
Public_Transport_Accessibility    0
Parking_Space                     0
Security                          0
Amenities                         0
Facing                            0
Owner_Type                        0
Availability_Status               0
dtype: int64

In [6]:
# remove duplicates
df.drop_duplicates(inplace=True)

In [7]:
df.dropna(subset=['Price_in_Lakhs', 'Size_in_SqFt'], inplace=True)

In [8]:
# Fill missing values

for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [9]:
# Feature Engineering
df['Age_of_Property']    = 2024 - df['Year_Built']
if 'Price_per_SqFt' not  in df.columns:
    df['Price_per_SqFt'] = (df['Price_in_Lakhs'] * 100000) / df['Size_in_SqFt']
df['School_Density_Score']= df['Nearby_Schools'] / df['Nearby_Schools'].max()
df['Amenity_Score']       = df['Amenities'].apply(
    lambda x: len(str(x).split(',')) if pd.notnull(x) else 0)
city_median               = df.groupby('City')['Price_in_Lakhs'].transform('median')
df['City_Price_Tier']     = pd.cut(city_median, bins=3, labels=['Low','Mid','High'])

In [12]:
# Target variables
city_med  = df.groupby('City')['Price_in_Lakhs'].transform('median')
med_psf   = df['Price_per_SqFt'].median()
df['Good_Investment'] = (
    ((df['Price_in_Lakhs'] <= city_med) & (df['Price_per_SqFt'] <= med_psf)) |
    ((df['BHK'] >= 3) & (df['Availability_Status'].str.lower() == 'available'))
).astype(int)
print("Good Investment %:", round(df['Good_Investment'].mean()*100,1))
 
df['Appreciation_Rate'] = df['City_Price_Tier'].astype(str).map(
    {'High':0.10,'Mid':0.08,'Low':0.06}).fillna(0.08)
df['Future_Price_5Y']   = df['Price_in_Lakhs'] * (1 + df['Appreciation_Rate'].astype(float))**5

Good Investment %: 39.9


In [13]:
# Outlier Removal
for col in ['Price_in_Lakhs', 'Size_in_SqFt']:
    Q1,Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1-1.5*IQR) & (df[col] <= Q3+1.5*IQR)]

print("Rows after outlier removal:", len(df))

Rows after outlier removal: 250000


In [14]:
 # Encode Categorical features
le = LabelEncoder()
for col in ['Property_Type','Furnished_Status','Facing','Owner_Type',
           'Availability_Status','Security','City_Price_Tier']:
    if col in df.columns:
        df[col+'_price_enc'] = df.groupby(col)['Price_in_Lakhs'].transform('mean')
df['Transport_enc'] = df['Public_Transport_Accessibility'].map(
    {'Low':0,'Medium':1,'High':2}).fillna(1)


In [17]:
# Scale Features
TARGET_COLS = ['Good_Investment','Future_Price_5Y']
DROP_ORIGINALS = ['Property_Type','Furnished_Status','Facing','Owner_Type',
                 'Availability_Status','Security','City_Price_Tier',
                 'state','City','Locality','Amenities','Year_Built','Appreciation_Rate']
df_model     = df.drop(columns=[c for c in DROP_ORIGINALS if c in df.columns],errors='ignore')
feature_cols = [c for c in df_model.columns
                if c not in TARGET_COLS and df_model[c].dtype in [np.float64,np.int64]]
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(df_model[feature_cols])
X_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df_model.index)

In [18]:
# Save
pd.concat([X_df, df_model[TARGET_COLS]], axis=1).to_csv("cleaned_data.csv",     index=False)
df.to_csv("cleaned_data_raw.csv", index=False)
print("\n✅ Step 1 Done!")
print("   cleaned_data.csv     → model training")
print("   cleaned_data_raw.csv → EDA")



✅ Step 1 Done!
   cleaned_data.csv     → model training
   cleaned_data_raw.csv → EDA
